In [7]:
import pandas as pd
import numpy as np
import random
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# ==========================================
# 1. LOAD DATA & TRAIN THE AI MODEL
# ==========================================
# Load the CSV (make sure the file is in the same folder as this notebook)
DeepSeek_df = pd.read_csv("DeepSeek_df_clean.csv")

FEATURES = [
    'TyreLife', 'TyreLifePercent', 'Avg_TrackTemp', 'Avg_AirTemp', 
    'TrackTemp_Delta', 'Peak_TyreEnergy', 'Energy_Std', 
    'PaceTrend', 'Quad_Term', 'DegAcceleration', 
    'Compound', 'CircuitArchetype', 'StintPhase'
]
TARGET = 'T2_Deg_Target'

train_df = DeepSeek_df.dropna(subset=[TARGET])
X = train_df[FEATURES]
y = train_df[TARGET]

cat_features = ['Compound', 'CircuitArchetype', 'StintPhase']
num_features = [c for c in FEATURES if c not in cat_features]

preprocessor = ColumnTransformer([
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_features)
])

model = Pipeline([
    ("prep", preprocessor), 
    ("rf", RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1))
])
model.fit(X, y)
print("✅ AI Model Trained!")

# ==========================================
# 2. THE SIMULATOR ENGINE
# ==========================================
def simulate_full_race(race_name, driver_code, rival_code, target_pit_lap, rival_pit_lap, pit_loss=22.0):
    my_data = DeepSeek_df[(DeepSeek_df['EventName'] == race_name) & (DeepSeek_df['Driver'] == driver_code)].copy()
    rival_data = DeepSeek_df[(DeepSeek_df['EventName'] == race_name) & (DeepSeek_df['Driver'] == rival_code)].copy()
    
    if my_data.empty or rival_data.empty: return None
    total_laps = int(max(my_data['LapNumber'].max(), rival_data['LapNumber'].max()))
    
    all_laps = pd.DataFrame({'LapNumber': range(1, total_laps + 1)})
    my_data = all_laps.merge(my_data, on='LapNumber', how='left').ffill()
    rival_data = all_laps.merge(rival_data, on='LapNumber', how='left').ffill()

    my_base_pace = my_data['LapTimeSeconds'].min()
    rival_base_pace = rival_data['LapTimeSeconds'].min()
    
    my_time, rival_time = 0.0, 0.0
    my_wear, rival_wear = 0.0, 0.0
    my_tyre_age, rival_tyre_age = 0, 0
    gap = 1.0 
    
    safety_car_lap = random.randint(10, total_laps) if random.random() < 0.20 else None
    
    for lap in range(1, total_laps + 1):
        my_row = my_data[my_data['LapNumber'] == lap].iloc[0]
        my_features = pd.DataFrame([{
            'TyreLife': my_tyre_age, 'TyreLifePercent': my_tyre_age / 25.0,
            'Avg_TrackTemp': my_row['Avg_TrackTemp'], 'Avg_AirTemp': my_row['Avg_AirTemp'],
            'TrackTemp_Delta': my_row['TrackTemp_Delta'], 'Peak_TyreEnergy': my_row['Peak_TyreEnergy'],
            'Energy_Std': my_row['Energy_Std'], 'PaceTrend': my_row['PaceTrend'],
            'Quad_Term': my_row['Quad_Term'], 'DegAcceleration': my_row['DegAcceleration'],
            'Compound': my_row['Compound'], 'CircuitArchetype': my_row['CircuitArchetype'],
            'StintPhase': 'Early' if my_tyre_age <= 5 else ('Mid' if my_tyre_age <= 15 else 'Late')
        }])
        my_wear_next = max(0, model.predict(my_features)[0])
        my_lap_time = my_base_pace + my_wear

        rival_row = rival_data[rival_data['LapNumber'] == lap].iloc[0]
        rival_features = pd.DataFrame([{
            'TyreLife': rival_tyre_age, 'TyreLifePercent': rival_tyre_age / 25.0,
            'Avg_TrackTemp': rival_row['Avg_TrackTemp'], 'Avg_AirTemp': rival_row['Avg_AirTemp'],
            'TrackTemp_Delta': rival_row['TrackTemp_Delta'], 'Peak_TyreEnergy': rival_row['Peak_TyreEnergy'],
            'Energy_Std': rival_row['Energy_Std'], 'PaceTrend': rival_row['PaceTrend'],
            'Quad_Term': rival_row['Quad_Term'], 'DegAcceleration': rival_row['DegAcceleration'],
            'Compound': rival_row['Compound'], 'CircuitArchetype': rival_row['CircuitArchetype'],
            'StintPhase': 'Early' if rival_tyre_age <= 5 else ('Mid' if rival_tyre_age <= 15 else 'Late')
        }])
        rival_wear_next = max(0, model.predict(rival_features)[0])
        rival_lap_time = rival_base_pace + rival_wear
        
        if lap == rival_pit_lap:
            rival_lap_time += pit_loss
            rival_tyre_age = 1
            rival_wear = 0.0
        else:
            rival_tyre_age += 1
            rival_wear += rival_wear_next

        actual_pit_loss = pit_loss
        if safety_car_lap and lap == safety_car_lap:
            actual_pit_loss = 10.0 
            
        if lap == target_pit_lap:
            my_lap_time += actual_pit_loss
            my_tyre_age = 1
            my_wear = 0.0
        else:
            my_tyre_age += 1
            my_wear += my_wear_next
            
        if 0 < (gap + (my_lap_time - rival_lap_time)) < 1.5:
            my_lap_time += 0.3 
            
        my_time += my_lap_time
        rival_time += rival_lap_time
        gap = rival_time - my_time 

    return my_time, rival_time, gap

# ==========================================
# 3. H1: THE OBJECTIVE FUNCTION & TEST
# ==========================================
def calculate_points(gap):
    if gap > 0: return 25, "P1" 
    else: return 18, "P2" 

RACE = "Bahrain Grand Prix"
ME = "HAM"
RIVAL = "VER"

print("\n🔄 Running H1 Optimization (Maximizing Points)...\n")

# Test pitting on Lap 24 (Undercut), 25 (Same), and 26 (Overcut)
for my_pit in [24, 25, 26]:
    my_time, rival_time, gap = simulate_full_race(RACE, ME, RIVAL, target_pit_lap=my_pit, rival_pit_lap=25)
    
    points, position = calculate_points(gap)
    
    print(f"Strategy: Pit on Lap {my_pit}")
    print(f"Result: We finish {position}. Points: {points}")
    if gap > 0:
        print(f"   (We beat VER by {abs(gap):.2f} seconds)\n")
    else:
        print(f"   (We lost to VER by {abs(gap):.2f} seconds)\n")

✅ AI Model Trained!

🔄 Running H1 Optimization (Maximizing Points)...

Strategy: Pit on Lap 24
Result: We finish P2. Points: 18
   (We lost to VER by 5.94 seconds)

Strategy: Pit on Lap 25
Result: We finish P2. Points: 18
   (We lost to VER by 4.58 seconds)

Strategy: Pit on Lap 26
Result: We finish P2. Points: 18
   (We lost to VER by 4.26 seconds)



# ----------- UNDERCUT / OVERCUT ENGINE ----------------

In [ ]:
# ==========================================
# H2: THE UNDERCUT / OVERCUT ENGINE
# ==========================================
def test_undercut_overcut(race_name, me, rival, rival_pit_lap):
    print(f"🏎️ Testing Reactions: Rival ({rival}) pits on Lap {rival_pit_lap}")
    print("-" * 50)
    
    # We will test 3 scenarios against the rival
    scenarios = {
        "Undercut (Pit 1 lap early)": rival_pit_lap - 1,
        "Same Strategy (Pit same lap)": rival_pit_lap,
        "Overcut (Pit 1 lap late)": rival_pit_lap + 1
    }
    
    best_action = ""
    best_gap = -999
    
    for action_name, my_pit_lap in scenarios.items():
        # Run the simulator
        result = simulate_full_race(race_name, me, rival, target_pit_lap=my_pit_lap, rival_pit_lap=rival_pit_lap)
        
        if result:
            
            my_time, rival_time, gap = result
            points, position = calculate_points(gap)
            
            if gap > 0:
                outcome = f"WIN by {abs(gap):.2f}s ({position})"
            else:
                outcome = f"LOSE by {abs(gap):.2f}s ({position})"
                
            print(f"Action: {action_name.ljust(30)} -> {outcome}")
            
            # Track the best action
            if gap > best_gap:
                best_gap = gap
                best_action = action_name
                
    print("-" * 50)
    print(f"🏆 BEST COUNTER-MOVE: {best_action}\n")

# Run H2
RACE = "Bahrain Grand Prix"
ME = "HAM"
RIVAL = "VER"

# Scenario: Verstappen pits on Lap 25. How do we react?
test_undercut_overcut(RACE, ME, RIVAL, rival_pit_lap=25)

# Scenario: What if Verstappen pits super early on Lap 15?
test_undercut_overcut(RACE, ME, RIVAL, rival_pit_lap=15)

🏎️ Testing Reactions: Rival (VER) pits on Lap 25
--------------------------------------------------
Action: Undercut (Pit 1 lap early)     -> LOSE by 5.94s (P2)
Action: Same Strategy (Pit same lap)   -> LOSE by 4.58s (P2)
Action: Overcut (Pit 1 lap late)       -> LOSE by 4.26s (P2)
--------------------------------------------------
🏆 BEST COUNTER-MOVE: Overcut (Pit 1 lap late)

🏎️ Testing Reactions: Rival (VER) pits on Lap 15
--------------------------------------------------
Action: Undercut (Pit 1 lap early)     -> LOSE by 7.75s (P2)
Action: Same Strategy (Pit same lap)   -> LOSE by 5.40s (P2)
Action: Overcut (Pit 1 lap late)       -> LOSE by 3.62s (P2)
--------------------------------------------------
🏆 BEST COUNTER-MOVE: Overcut (Pit 1 lap late)

